In [1]:
import os
import gc
import math
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats as stats
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from concurrent.futures import ProcessPoolExecutor

In [2]:
def interpolate_data(file_path):
    df = pd.read_csv(file_path, sep='\t')
    participant_ids = df['id_participant'].unique()
    result_list = []
    for participant_id in participant_ids:
        participant_data = df[df['id_participant'] == participant_id]
        x = participant_data['days_from_start']
        y = participant_data['score']
        f = interp1d(x, y, kind='linear', bounds_error=False, fill_value=(y.min(), y.max()))
        max_days = participant_data['days_from_start'].max()
        days = np.arange(0, 2000 + 50, 50)
        predicted_scores = f(days)
        for day, score in zip(days, predicted_scores):
            result_list.append({'participant_id': participant_id, 'days': day, 'interpolated_PDDS': score})
    results = pd.DataFrame(result_list)
    return results

def get_dict_from_threshold(data, index, threshold):
    data = data.drop(columns='key')
    row = data.iloc[index]
    columns = data[0] if isinstance(data, list) else data.columns
    result_dict = {col: (1 if row.iloc[i] > threshold else 0) for i, col in enumerate(columns) if not math.isnan(row.iloc[i])}
    num_ones = sum(value == 1 for value in result_dict.values())
    num_zeros = sum(value == 0 for value in result_dict.values())
    return result_dict, num_ones, num_zeros

def get_key(data, index):
    row = data.iloc[index]['key']
    return row

def get_index(data, key):
    index = data[data['key'] == key].index[0]
    return int(index)

def get_value(data, key, pid):
    return data.iloc[get_index(data, key)][pid]

def plot_average_scores(data, pdds, class_dict, name, threshold=15):
    num_ones = sum(value == 1 for value in class_dict.values())
    num_zeros = sum(value == 0 for value in class_dict.values())
    path = f"./good_dmrs/{name}.pdf"
    
    if not os.path.exists(path) and num_ones >= threshold and num_zeros >= threshold:
        pdds['binary_class'] = pdds['Participant Name'].map(class_dict)
        
        stratified_pdds = pdds.groupby(['Days from Blood Draw', 'binary_class']).agg(
            mean=('interpolated_PDDS', 'mean'),
            std=('interpolated_PDDS', 'std'),
            count=('interpolated_PDDS', 'size')
        ).reset_index()
        stratified_pdds['ci95'] = stratified_pdds['std'] / np.sqrt(stratified_pdds['count']) * 1.96
        
        fig, (ax_box, ax_line) = plt.subplots(2, 1, figsize=(5, 6), dpi=300, gridspec_kw={'height_ratios': [1, 1]})
        
        days_intervals = range(0, 2000 + 1, 250)
        significant_days = []
        absolute_difference = []
        
        for day in days_intervals:
            day_data = pdds[pdds['Days from Blood Draw'] == day]
            class_0 = day_data[day_data['binary_class'] == 0]['interpolated_PDDS']
            class_1 = day_data[day_data['binary_class'] == 1]['interpolated_PDDS']
            
            if len(class_0) > 1 and len(class_1) > 1:
                t_stat, p_val = stats.ttest_ind(class_0, class_1, equal_var=False)
                if p_val*len(days_intervals) < 0.05:
                    significant_days.append(day)
                    absolute_difference.append(abs(class_0.mean()-class_1.mean()))
                    ax_line.text(day, stratified_pdds['mean'].max()+0.2, '*', color='black', ha='center', fontsize=8)
                else:
                    significant_days.append(0)
            sns.violinplot(data=day_data, x='Days from Blood Draw', y='interpolated_PDDS', hue='binary_class', ax=ax_box, palette=['teal', 'darkorchid'], split=True, inner="quart", linewidth=0.5)
        
        if sum(significant_days)==0:
            return
        
        ax_box.set_ylabel('PDDS Score', fontsize=8)
        ax_box.set_title(f"{name} | Purple: n={num_ones}, Teal: n={num_zeros}", fontsize=10)
        ax_box.set_ylim(bottom=0, top=11)
        ax_box.tick_params(axis='both', which='major', labelsize=8)
        ax_box.legend().remove()
        ax_box.set_xlabel('')

        for binary_class, color in zip([0, 1], ['teal', 'darkorchid']):
            class_stats = stratified_pdds[stratified_pdds['binary_class'] == binary_class]
            sns.lineplot(data=class_stats, x='Days from Blood Draw', y='mean', color=color, linewidth=0.5, ax=ax_line)
            ax_line.fill_between(class_stats['Days from Blood Draw'], 
                                 class_stats['mean'] - class_stats['ci95'], 
                                 class_stats['mean'] + class_stats['ci95'],
                                 color=color, alpha=0.05)
        
        ax_line.set_xlabel('Days from Blood Draw', fontsize=8)
        ax_line.set_ylabel('Mean PDDS Score', fontsize=8)
        ax_line.tick_params(axis='both', which='major', labelsize=8)
        ax_line.set_xlim(left=-120, right=2120)
        ax_line.set_ylim(bottom=0)
        
        plt.tight_layout()
        plt.savefig(path)
        plt.close()
        return path, significant_days, absolute_difference
        
def generate_row_colors(species_list, samples):
    row_colors = {}
    for col in species_list:
        species = samples[col]
        lut = dict(zip([0, 1, 2], ["teal", "darkorchid", "white"]))
        row_colors[col] = species.map(lut)
    return pd.DataFrame(row_colors)

In [3]:
dmr_data = pd.read_csv("./data/dmr_data.txt", sep='\t')
samples = interpolate_data("./data/survival_pdds_days.tsv")
participants_to_drop = ['PRT180632', 'PRT190871', 'PRT190912', 'PRT190945']
samples = samples[~samples['participant_id'].isin(participants_to_drop)]
df = samples
df.rename(columns={'participant_id': 'Participant Name', 'days': 'Days from Blood Draw'}, inplace=True)
pivot_df = df.pivot_table(index='Participant Name', columns='Days from Blood Draw', values='interpolated_PDDS', aggfunc='first')

In [ ]:
def task(i, j, dmr_data, samples):
    j = round(j, 2)
    class_dict = get_dict_from_threshold(dmr_data, i, j)[0]
    name = str(get_key(dmr_data, i)) + str(",") + str(j)
    return plot_average_scores(dmr_data, samples, class_dict, name)

def run_multithreaded(dmr_data, samples):
    paths = []
    significant_days_list = []
    absolute_differences = []

    with ProcessPoolExecutor(max_workers=128) as executor:
        futures = []
        for i in range(512):
            for j in np.arange(0, 1, 0.05):
                futures.append(executor.submit(task, i, j, dmr_data, samples))

        for future in futures:
            result = future.result()
            if result is not None:
                path, significant_days, absolute_difference = result
                paths.append(path)
                significant_days_list.append(significant_days)
                absolute_differences.append(absolute_difference)

    return paths, significant_days_list, absolute_differences

paths, significant_days_list, absolute_differences = run_multithreaded(dmr_data, samples)

In [4]:
data = pd.read_csv('./data/good_dmrs_w_names.bed', sep='\t')
results = list(zip(data.iloc[:, 0], data.iloc[:, 2].astype(float)))
for result in results:
    i = get_index(dmr_data, result[0])
    j = result[1]
    class_dict = get_dict_from_threshold(dmr_data, i, j)[0]
    name = str(get_key(dmr_data, i)) + str(" | ") + str(j)
    plot_average_scores(dmr_data, samples, class_dict, name)

In [5]:
rows = []
row_label = pd.DataFrame()
for result in results:
    key = result[0]
    rows.append(key)
    row_label[key] = None
    threshold = result[1]
    for index, row in pivot_df.iterrows():  
        dmr_val = get_value(dmr_data, key, index)
        row_label.at[index, key] = int(dmr_val>threshold) if not math.isnan(dmr_val) else 2